# CNN model

## prepare and load data

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = np.load("processed_power_data.npz")

X_train = torch.tensor(data["X_train"], dtype=torch.float32)
y_train = torch.tensor(data["y_train"], dtype=torch.float32)

X_val   = torch.tensor(data["X_val"], dtype=torch.float32)
y_val   = torch.tensor(data["y_val"], dtype=torch.float32)

X_test  = torch.tensor(data["X_test"], dtype=torch.float32)
y_test  = torch.tensor(data["y_test"], dtype=torch.float32)


Using device: cuda


# Data loader

In [8]:
BATCH_SIZE = 64

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [9]:
from CNN import CNNTimeSeries

INPUT_DIM  = X_train.shape[-1]
OUTPUT_DIM = y_train.shape[-1]

model = CNNTimeSeries(INPUT_DIM, OUTPUT_DIM).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


train and check iteration

In [ ]:
EPOCHS = 20

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    # -------- Training --------
    model.train()
    train_loss = 0.0
    
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * xb.size(0)
    
    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)
    
    # -------- Validation --------
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    
    val_loss /= len(val_loader.dataset)
    val_losses.append(val_loss)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


Epoch [1/20] Train Loss: 0.8634 | Val Loss: 0.7195
Epoch [2/20] Train Loss: 0.7900 | Val Loss: 0.7180
Epoch [3/20] Train Loss: 0.7828 | Val Loss: 0.7026
Epoch [4/20] Train Loss: 0.7743 | Val Loss: 0.6961
Epoch [5/20] Train Loss: 0.7661 | Val Loss: 0.6949
Epoch [6/20] Train Loss: 0.7585 | Val Loss: 0.6882
Epoch [7/20] Train Loss: 0.7518 | Val Loss: 0.6789
Epoch [8/20] Train Loss: 0.7453 | Val Loss: 0.6761
Epoch [9/20] Train Loss: 0.7393 | Val Loss: 0.6665
Epoch [10/20] Train Loss: 0.7344 | Val Loss: 0.6651
Epoch [11/20] Train Loss: 0.7320 | Val Loss: 0.6656
Epoch [12/20] Train Loss: 0.7294 | Val Loss: 0.6613
Epoch [13/20] Train Loss: 0.7271 | Val Loss: 0.6673
Epoch [14/20] Train Loss: 0.7255 | Val Loss: 0.6620
Epoch [15/20] Train Loss: 0.7236 | Val Loss: 0.6575
Epoch [16/20] Train Loss: 0.7215 | Val Loss: 0.6698
Epoch [17/20] Train Loss: 0.7205 | Val Loss: 0.6612
Epoch [18/20] Train Loss: 0.7185 | Val Loss: 0.6573
Epoch [19/20] Train Loss: 0.7173 | Val Loss: 0.6593
Epoch [20/20] Train L

: 

## Loss curve and visualization

In [ ]:
import matplotlib.pyplot as plt
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.title("CNN Training and Validation Loss")
plt.show()


## evaluate

In [ ]:
model.eval()
preds_list = []
true_list = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds = model(xb).cpu().numpy()
        preds_list.append(preds)
        true_list.append(yb.numpy())

y_pred = np.concatenate(preds_list, axis=0)
y_true = np.concatenate(true_list, axis=0)

from sklearn.metrics import mean_absolute_error, mean_squared_error

mae  = mean_absolute_error(y_true.flatten(), y_pred.flatten())
rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))

print(f"CNN Test MAE : {mae:.4f}")
print(f"CNN Test RMSE: {rmse:.4f}")

idx = 0  # 第一个测试样本

plt.plot(y_true[idx], label="True")
plt.plot(y_pred[idx], label="Predicted")
plt.xlabel("Time (min)")
plt.ylabel("Load")
plt.title("CNN Forecast vs Ground Truth")
plt.legend()
plt.show()
